## torch.Tensor
- requires_grad
- grad
- grad_fn
- backward()

In [1]:
import torch

In [20]:
x = torch.tensor(2.0, requires_grad = True) # default is False
# meaning: do we need to cal the gradien for this tensor -> true: pytorch follow
x # x laf original tensor ( leaf tensor ), grad_fn = None

tensor(2., requires_grad=True)

In [26]:
y = x * x # tensor y was born from x, relationship is: y = x*x = x^2
y.grad_fn # pytorch known that y born from multiply
y

tensor(4., grad_fn=<MulBackward0>)

In [27]:
y.backward() # derivate dy/dx => x^2 -> 2x
y

tensor(4., grad_fn=<MulBackward0>)

In [24]:
x.grad # The result of dy/dx at x = 2.0

tensor(4.)

## Build model 

In [28]:
import torch 
import torch.nn as nn  # self-study block
import torch.nn.functional as F # function for some works like: relu, pool,..

### Convolutional Neural Network
- Receive the grey image (ex: Mnist 28x28) -> extract features -> classifier 10 layers (0-9)

In [37]:
class Net(nn.Module):
    def __init__(self): # declare layers
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(1, 6, (3,3)) # 1 feature map, 6 filter, kernel 3x3
        self.conv2 = nn.Conv2d(6, 16, (3,3))

        self.fc1 = nn.Linear(16*6*6, 120) # fully-connected layer
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x): # data flows
            x = self.conv1(x) #extract feature, output í 6 feature maps
            x = F.relu(x) # find max (0,x)
            x = F.max_pool2d(x, (2,2)) # down the size, keep important data, info

            #kernel: w*h
            x = self.conv2(x)
            x = F.relu(x)
            x = F.max_pool2d(x, (2,2))
            #tensor: x.shape(1, 16, 6, 6)
        
            x = x.view(-1,  self.num_flat_features(x))
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = self.fc3(x)
            
            return x
    def num_flat_features(self, x ): # caluculate the neccasary neurons
            size = x.size()[1:] #drop batch size 
            num_features = 1
            for s in size:
                num_features *= s
            return num_features
net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=576, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [ ]:
input_image = torch.rand(1, 1, 32, 32) # 1 image, 1 channel (grey), 32x32
output = net(input_image)
output.size()

Ban đầu ta có 1 image, 1 channel (1 lớp thông tin) kích thước 32×32.
Ảnh được đưa qua lớp convolution đầu tiên với 6 kernel 3×3, tạo ra 6 feature maps kích thước 30×30.
Các feature maps này đi qua ReLU để loại bỏ giá trị âm và qua pooling để giảm kích thước còn 15×15.
Tiếp theo, mạng sử dụng lớp convolution thứ hai với 16 kernel để trích xuất đặc trưng phức tạp hơn, sau pooling còn 16 feature maps kích thước 6×6.
Các feature maps này được flatten thành vector gồm 576 neuron ( xếp 36 neuron liên tục 16 lần ) và đưa vào các fully connected layers (giảm số neuron qua từng lớp 576 - 120 - 84 - 10) để học mối quan hệ giữa đặc trưng và nhãn, cuối cùng tạo ra 10 giá trị đầu ra đại diện cho các lớp.
Mạng chọn lớp có xác suất cao nhất làm kết quả dự đoán.

## build model with nn.sequential

In [1]:
import torch
import torch.nn as nn

In [2]:
net = nn.Sequential()

In [9]:
class Flatten(nn.Module):
    def forward(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return x.view(-1, num_features)

In [13]:
net.add_module("Convl", nn.Conv2d(1, 6, (3,3)))
net.add_module("reLu", nn.ReLU())
net.add_module("Maxpooling1", nn.MaxPool2d(2,2))

net.add_module("Conv2", nn.Conv2d(6, 16, (3,3)))
net.add_module("reLu", nn.ReLU())
net.add_module("Maxpooling2", nn.MaxPool2d(2,2))

net.add_module("Flatten", Flatten())

net.add_module("Fc1", nn.Linear(16*6*6, 120))
net.add_module("Fc2", nn.Linear(120, 84))
net.add_module("Fc3", nn.Linear(84, 10))


In [11]:
print(net)

Sequential(
  (Convl): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (reLu): ReLU()
  (Maxpooling1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (Maxpooling2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Flatten): Flatten()
  (Fc1): Linear(in_features=84, out_features=10, bias=True)
)


In [14]:
input = torch.rand(1, 1, 32, 32)
output = net(input)

In [15]:
print(output)

tensor([[-0.1270,  0.1154,  0.0138,  0.0395, -0.0639, -0.1623, -0.1518, -0.0774,
         -0.0386,  0.0338]], grad_fn=<AddmmBackward0>)


In [17]:
print(output.size())

torch.Size([1, 10])
